# Notebook 03 — CNN Classification Training
**Thesis: Computer Vision and Deep Learning for Real-Time Quality Inspection in Manufacturing**

## Overview
Train CNN classifiers (ResNet-50, EfficientNet-B0, MobileNetV3-Large) on three industrial defect datasets:
- **NEU-DET**: 6 defect classes (classification on cropped patches from bboxes)
- **DAGM 2007**: Binary classification (defective / non_defective)
- **KSDD2**: Binary classification (defective / non_defective, weighted loss for imbalance)

## Pipeline
1. Load classification data from Drive (ImageFolder format, output of Notebook 01)
2. Apply torchvision transforms with augmentation
3. Handle class imbalance (KSDD2): WeightedRandomSampler + class-weighted CrossEntropyLoss
4. Transfer learning: load ImageNet pretrained weights, replace head
5. Two-stage training: freeze backbone (Stage 1), fine-tune all layers (Stage 2)
6. AdamW optimiser, CosineAnnealingLR scheduler, early stopping (patience=10)
7. Evaluate: Accuracy, Macro-F1, Confusion Matrix, Classification Report per dataset
8. Export all 9 trained models to ONNX
9. Save results and checkpoint to Drive

## Models
| Model | Params | Notes |
|-------|--------|-------|
| ResNet-50 | ~25M | Baseline — established benchmark |
| EfficientNet-B0 | ~5M | Efficient scaling |
| MobileNetV3-Large | ~5M | Mobile-optimized |

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────────
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install pandas scikit-learn matplotlib seaborn tqdm openpyxl -q
!pip install onnx onnxruntime -q

from google.colab import drive
drive.mount('/content/drive')

import os, random, json, datetime
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DRIVE_ROOT = '/content/drive/MyDrive/thesis_project'

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Drive root: {DRIVE_ROOT}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 102.3 MB/s eta 0:00:00
Mounted at /content/drive
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
Drive root: /content/drive/MyDrive/thesis_project


In [ ]:
# ── Cell 2: Verify Notebook 01 outputs ────────────────────────────────────────
import pathlib

ckpt_path = pathlib.Path(DRIVE_ROOT) / 'checkpoints/notebook01_status.json'
if not ckpt_path.exists():
    raise FileNotFoundError('Run Notebook 01 first!')

status = json.loads(ckpt_path.read_text())
print('Notebook 01 status loaded.')

DATASETS = ['NEU-DET', 'DAGM', 'KSDD2']
for ds in DATASETS:
    cls_dir = pathlib.Path(DRIVE_ROOT) / f'datasets/{ds}/classification'
    for split in ['train', 'val', 'test']:
        p = cls_dir / split
        n = sum(len(list(p.glob('**/*.jpg'))) + len(list(p.glob('**/*.png'))) for _ in [1])
        print(f'  {ds}/{split}: ~{n} images')

Notebook 01 status loaded.
  NEU-DET/train: ~3620 images
  NEU-DET/val: ~766 images
  NEU-DET/test: ~805 images
  DAGM/train: ~3616 images
  DAGM/val: ~771 images
  DAGM/test: ~766 images
  KSDD2/train: ~4669 images
  KSDD2/val: ~545 images
  KSDD2/test: ~501 images


In [ ]:
# ── Cell 3: Data Transforms (torchvision) ─────────────────────────────────────
from torchvision import transforms

TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

VAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Transforms defined.')

Transforms defined.


In [ ]:
# ── Cell 4: Training Hyperparameters ─────────────────────────────────────────
CNN_CONFIG = {
    'epochs': 50,
    'batch_size': 32,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'patience': 10,
    'step_lr_step': 10,
    'step_lr_gamma': 0.5,
    'seed': SEED,
    'freeze_epochs': 5,   # Stage 1: freeze backbone
    'unfreeze_lr': 1e-5,  # Stage 2: lower LR after unfreeze
}
CNN_MODELS = ['resnet50', 'efficientnet_b0', 'mobilenet_v3_large']
DATASETS_CNN = ['NEU-DET', 'DAGM', 'KSDD2']
print('CNN Config:', CNN_CONFIG)

CNN Config: {'epochs': 50, 'batch_size': 32, 'lr': 0.0001, 'weight_decay': 0.0001, 'patience': 10, 'step_lr_step': 10, 'step_lr_gamma': 0.5, 'seed': 42, 'freeze_epochs': 5, 'unfreeze_lr': 1e-05}


In [ ]:
# ── Cell 5: DataLoader Factory ────────────────────────────────────────────────
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

def make_loaders(dataset_name, batch_size=32):
    base = f'{DRIVE_ROOT}/datasets/{dataset_name}/classification'

    train_ds = ImageFolder(f'{base}/train', transform=TRAIN_TRANSFORMS)
    val_ds   = ImageFolder(f'{base}/val',   transform=VAL_TRANSFORMS)
    test_ds  = ImageFolder(f'{base}/test',  transform=VAL_TRANSFORMS)

    class_names = train_ds.classes
    num_classes = len(class_names)
    print(f'  {dataset_name}: {num_classes} classes: {class_names}')
    print(f'  Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

    # Compute class weights from training set (used for loss and sampler)
    targets = torch.tensor(train_ds.targets)
    class_counts = torch.bincount(targets)
    class_weights = (1.0 / class_counts.float()).to(DEVICE)
    class_weights = class_weights / class_weights.sum() * num_classes  # normalize

    # For KSDD2 (heavily imbalanced): use WeightedRandomSampler
    if dataset_name == 'KSDD2':
        sample_weights = class_weights.cpu()[targets]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                                  num_workers=2, pin_memory=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                  num_workers=2, pin_memory=True)

    val_loader  = DataLoader(val_ds,  batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader, class_names, class_weights

print('make_loaders() defined.')

make_loaders() defined.


In [ ]:
# ── Cell 6: Model Factory (Transfer Learning) ─────────────────────────────────
import torchvision.models as models
import torch.nn as nn

def build_model(model_name: str, num_classes: int) -> nn.Module:
    """Load ImageNet pretrained backbone and replace classification head."""
    if model_name == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'mobilenet_v3_large':
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    else:
        raise ValueError(f'Unknown model: {model_name}')
    return m.to(DEVICE)

def freeze_backbone(model, model_name):
    """Freeze all layers except the classification head."""
    for param in model.parameters():
        param.requires_grad = False
    # Unfreeze head only
    if model_name == 'resnet50':
        for param in model.fc.parameters(): param.requires_grad = True
    elif model_name == 'efficientnet_b0':
        for param in model.classifier.parameters(): param.requires_grad = True
    elif model_name == 'mobilenet_v3_large':
        for param in model.classifier.parameters(): param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

print('build_model() and freeze helpers defined.')

build_model() and freeze helpers defined.


In [ ]:
# ── Cell 7: Training & Evaluation Functions ───────────────────────────────────
import time
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return running_loss / total, acc, f1, all_preds, all_labels

@torch.no_grad()
def get_probabilities(model, loader):
    """Get softmax probabilities for ROC/PR curves."""
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        probs = torch.softmax(model(imgs), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.extend(labels.numpy())
    return np.vstack(all_probs), np.array(all_labels)

print('Training helpers defined.')

Training helpers defined.


In [ ]:
# ── Cell 8: Main Training Loop (all 9 models) ─────────────────────────────────
import pandas as pd
import numpy as np
import os, shutil

results_log = []

for dataset in DATASETS_CNN:
    print(f'\n{"="*70}')
    print(f'DATASET: {dataset}')
    print(f'{"="*70}')

    train_loader, val_loader, test_loader, class_names, class_weights = make_loaders(
        dataset, CNN_CONFIG['batch_size']
    )
    num_classes = len(class_names)

    for model_name in CNN_MODELS:
        run_name = f'{dataset}_{model_name}'
        save_path = f'{DRIVE_ROOT}/models/cnn/full/{run_name}_best.pth'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        # --- Checkpoint-resume: skip if already trained ---
        if os.path.exists(save_path):
            print(f'\n[SKIP] {run_name} already trained.')
            results_log.append({'dataset': dataset, 'model': model_name, 'status': 'skipped'})
            continue

        print(f'\n--- Training {run_name} ---')
        t0 = time.time()

        model = build_model(model_name, num_classes)
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

        # ── Stage 1: Train head only (freeze backbone) ──
        freeze_backbone(model, model_name)
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CNN_CONFIG['lr'], weight_decay=CNN_CONFIG['weight_decay']
        )

        print(f'  Stage 1: Training head for {CNN_CONFIG["freeze_epochs"]} epochs...')
        for epoch in range(CNN_CONFIG['freeze_epochs']):
            tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
            vl_loss, vl_acc, vl_f1, _, _ = eval_epoch(model, val_loader, criterion)
            print(f'    Epoch {epoch+1}/{CNN_CONFIG["freeze_epochs"]} | '
                  f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | '
                  f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} vl_f1={vl_f1:.4f}')

        # ── Stage 2: Fine-tune all layers ──
        unfreeze_all(model)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CNN_CONFIG['unfreeze_lr'], weight_decay=CNN_CONFIG['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CNN_CONFIG['epochs'] - CNN_CONFIG['freeze_epochs']
        )

        best_val_f1 = 0.0
        patience_counter = 0
        epoch_log = []

        print(f'  Stage 2: Fine-tuning all layers for up to {CNN_CONFIG["epochs"]} epochs...')
        for epoch in range(CNN_CONFIG['epochs']):
            tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
            vl_loss, vl_acc, vl_f1, vl_preds, vl_labels = eval_epoch(model, val_loader, criterion)
            scheduler.step()

            epoch_log.append({
                'epoch': epoch + 1,
                'train_loss': round(tr_loss, 5),
                'train_acc': round(tr_acc, 5),
                'val_loss': round(vl_loss, 5),
                'val_acc': round(vl_acc, 5),
                'val_f1': round(vl_f1, 5),
            })

            if vl_f1 > best_val_f1:
                best_val_f1 = vl_f1
                torch.save(model.state_dict(), save_path)
                patience_counter = 0
                marker = ' <- best'
            else:
                patience_counter += 1
                marker = ''

            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f'    Epoch {epoch+1} | '
                      f'tr_acc={tr_acc:.4f} vl_acc={vl_acc:.4f} vl_f1={vl_f1:.4f}{marker}')

            if patience_counter >= CNN_CONFIG['patience']:
                print(f'    Early stopping at epoch {epoch+1} (patience={CNN_CONFIG["patience"]})')
                break

        # Save epoch log CSV
        log_dir = f'{DRIVE_ROOT}/training_logs/cnn/{run_name}'
        os.makedirs(log_dir, exist_ok=True)
        pd.DataFrame(epoch_log).to_csv(f'{log_dir}/epoch_log.csv', index=False)

        # ── Test set evaluation ──
        print(f'  Evaluating on test set...')
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
        _, test_acc, test_f1, test_preds, test_labels = eval_epoch(model, test_loader, criterion)

        # Confusion matrix
        cm = confusion_matrix(test_labels, test_preds)
        np.save(f'{log_dir}/confusion_matrix.npy', cm)

        # Per-class report
        report = classification_report(test_labels, test_preds, target_names=class_names, output_dict=True)
        pd.DataFrame(report).transpose().to_csv(f'{log_dir}/classification_report.csv')

        # Probabilities for ROC/PR curves (test set)
        test_probs, _ = get_probabilities(model, test_loader)
        np.save(f'{log_dir}/test_probs.npy', test_probs)
        np.save(f'{log_dir}/test_labels.npy', np.array(test_labels))

        elapsed = time.time() - t0
        entry = {
            'dataset': dataset,
            'model': model_name,
            'num_classes': num_classes,
            'test_accuracy': round(test_acc, 4),
            'test_f1_macro': round(test_f1, 4),
            'best_val_f1': round(best_val_f1, 4),
            'train_time_min': round(elapsed / 60, 1),
            'status': 'trained',
        }
        results_log.append(entry)
        print(f'  -> Test acc={test_acc:.4f} F1={test_f1:.4f} | time={elapsed/60:.1f}min')

        # Free GPU memory
        del model
        torch.cuda.empty_cache()

print('\nAll CNN models trained!')
df_results = pd.DataFrame(results_log)
os.makedirs(f'{DRIVE_ROOT}/results/tables', exist_ok=True)
df_results.to_csv(f'{DRIVE_ROOT}/results/tables/cnn_test_results.csv', index=False)
print(df_results.to_string())


DATASET: NEU-DET
  NEU-DET: 5 classes: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled_in_scale']
  Train: 3620, Val: 766, Test: 805

--- Training NEU-DET_resnet50 ---
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 168MB/s]


  Stage 1: Training head for 5 epochs...


    Epoch 1/5 | tr_loss=1.2886 tr_acc=0.6384 | vl_loss=0.9576 vl_acc=0.8368 vl_f1=0.8376


    Epoch 2/5 | tr_loss=0.8916 tr_acc=0.8337 | vl_loss=0.6864 vl_acc=0.8734 vl_f1=0.8719


...

    Epoch 30 | tr_acc=0.8996 vl_acc=0.9101 vl_f1=0.8137


    Epoch 35 | tr_acc=0.9130 vl_acc=0.9248 vl_f1=0.8369


    Epoch 40 | tr_acc=0.9128 vl_acc=0.9211 vl_f1=0.8309


    Epoch 45 | tr_acc=0.9092 vl_acc=0.9193 vl_f1=0.8280


    Epoch 50 | tr_acc=0.9045 vl_acc=0.9229 vl_f1=0.8339
  Evaluating on test set...
  -> Test acc=0.9381 F1=0.8673 | time=65.9min

All CNN models trained!
   dataset               model  num_classes  test_accuracy  test_f1_macro  best_val_f1  train_time_min   status
0  NEU-DET            resnet50            5         0.9988         0.9990       0.9958            34.9  trained
1  NEU-DET     efficientnet_b0            5         0.9851         0.9855       0.9958            24.5  trained
2  NEU-DET  mobilenet_v3_large            5         0.9863         0.9857       0.9937            13.8  trained
3     DAGM            resnet50            2         0.9948         0.9882       0.9971            47.8  trained
4     DAGM     efficientnet_b0            2         0.8629         0.7499       0.8022            27.9  trained
5     DAGM  mobilenet_v3_large            2         0.9634         0.9159       0.9104            32.2  trained
6    KSDD2            resnet50            2         0.9721   

In [ ]:
import os

print('Deleting previous ONNX files...')
for dataset in DATASETS_CNN:
    for model_name in CNN_MODELS:
        run_name = f'{dataset}_{model_name}'
        onnx_path = f'{DRIVE_ROOT}/models/cnn/onnx/{run_name}.onnx'
        if os.path.exists(onnx_path):
            os.remove(onnx_path)
            print(f'  Deleted: {onnx_path}')
print('Previous ONNX files deleted.')

Deleting previous ONNX files...
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/NEU-DET_resnet50.onnx
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/NEU-DET_efficientnet_b0.onnx
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/NEU-DET_mobilenet_v3_large.onnx
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/DAGM_resnet50.onnx
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/DAGM_efficientnet_b0.onnx
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/DAGM_mobilenet_v3_large.onnx
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_resnet50.onnx
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_efficientnet_b0.onnx
  Deleted: /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_mobilenet_v3_large.onnx
Previous ONNX files deleted.


In [ ]:
# ── Cell 9: ONNX Export (all 9 trained models) ────────────────────────────────
!pip install onnxscript -q

import torch
import torch.onnx
import onnx, os

def export_to_onnx(model_name, dataset, num_classes):
    run_name = f'{dataset}_{model_name}'
    src_path  = f'{DRIVE_ROOT}/models/cnn/full/{run_name}_best.pth'
    onnx_path = f'{DRIVE_ROOT}/models/cnn/onnx/{run_name}.onnx'
    os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

    if not os.path.exists(src_path):
        print(f'[SKIP] Model not found: {src_path}')
        return

    if os.path.exists(onnx_path):
        print(f'[SKIP] ONNX already exists: {onnx_path}')
        return

    model = build_model(model_name, num_classes)
    model.load_state_dict(torch.load(src_path, map_location='cpu'))
    model.eval().cpu()

    dummy = torch.randn(1, 3, 224, 224)
    torch.onnx.export(
        model, dummy, onnx_path,
        opset_version=18, # Changed opset_version from 13 to 18
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
        do_constant_folding=True,
    )

    # Verify
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    size_mb = os.path.getsize(onnx_path) / 1e6
    print(f'  Exported {run_name} -> {onnx_path} ({size_mb:.1f} MB)')

# Dynamically create num_classes_map from the results_log
# This assumes num_classes for a dataset is consistent across models
actual_num_classes_map = {}
for entry in results_log:
    if entry['dataset'] not in actual_num_classes_map:
        actual_num_classes_map[entry['dataset']] = entry['num_classes']

for dataset in DATASETS_CNN:
    num_classes = actual_num_classes_map[dataset]
    for model_name in CNN_MODELS:
        export_to_onnx(model_name, dataset, num_classes)

print('ONNX export complete.')

/tmp/ipython-input-2219678993.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0219 13:40:16.135000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:40:16.137000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:40:16.139000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0).

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 106 of general pattern rewrite rules.
  Exported NEU-DET_resnet50 -> /content/drive/MyDrive/thesis_project/models/cnn/onnx/NEU-DET_resnet50.onnx (0.2 MB)


/tmp/ipython-input-2219678993.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0219 13:40:25.510000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:40:25.511000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:40:25.513000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0).

[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 98 of general pattern rewrite rules.
  Exported NEU-DET_efficientnet_b0 -> /content/drive/MyDrive/thesis_project/models/cnn/onnx/NEU-DET_efficientnet_b0.onnx (0.6 MB)


/tmp/ipython-input-2219678993.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0219 13:40:32.528000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:40:32.529000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:40:32.532000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0).

...

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 106 of general pattern rewrite rules.
  Exported KSDD2_resnet50 -> /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_resnet50.onnx (0.2 MB)


/tmp/ipython-input-2219678993.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0219 13:41:08.880000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:41:08.881000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:41:08.883000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0).

[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 98 of general pattern rewrite rules.
  Exported KSDD2_efficientnet_b0 -> /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_efficientnet_b0.onnx (0.6 MB)


/tmp/ipython-input-2219678993.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0219 13:41:15.999000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:41:16.003000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 13:41:16.006000 301 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0).

[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 92 of general pattern rewrite rules.
  Exported KSDD2_mobilenet_v3_large -> /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_mobilenet_v3_large.onnx (0.3 MB)
ONNX export complete.


In [ ]:
# ── Cell 10: Checkpoint ───────────────────────────────────────────────────────
import json, datetime, os

checkpoint = {
    'notebook': '03_cnn_classification_training',
    'completed': True,
    'timestamp': datetime.datetime.now().isoformat(),
    'models_trained': results_log,
    'test_results_csv': f'{DRIVE_ROOT}/results/tables/cnn_test_results.csv',
    'models_dir': f'{DRIVE_ROOT}/models/cnn/',
    'logs_dir': f'{DRIVE_ROOT}/training_logs/cnn/',
}

os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
ckpt_path = f'{DRIVE_ROOT}/checkpoints/notebook03_status.json'
with open(ckpt_path, 'w') as f:
    json.dump(checkpoint, f, indent=2)

print('='*60)
print('Notebook 03 Complete!')
print(f'Checkpoint: {ckpt_path}')
print(f'Models: {DRIVE_ROOT}/models/cnn/full/')
print(f'ONNX: {DRIVE_ROOT}/models/cnn/onnx/')
print(f'Logs: {DRIVE_ROOT}/training_logs/cnn/')
print(f'Results: {DRIVE_ROOT}/results/tables/cnn_test_results.csv')
print('='*60)

Notebook 03 Complete!
Checkpoint: /content/drive/MyDrive/thesis_project/checkpoints/notebook03_status.json
Models: /content/drive/MyDrive/thesis_project/models/cnn/full/
ONNX: /content/drive/MyDrive/thesis_project/models/cnn/onnx/
Logs: /content/drive/MyDrive/thesis_project/training_logs/cnn/
Results: /content/drive/MyDrive/thesis_project/results/tables/cnn_test_results.csv
